# 06 — Spatial Subsetting with AOI Validation

The only subsetting method live here is a latitude/longitude bounding box. The scene's real extent gets reported in WGS84 first, then whatever AOI you request is checked against that extent before any data is read. An AOI that falls entirely outside the scene is stopped outright; one that partially overlaps is flagged and needs your confirmation.

Pixel/row-column and projected X/Y modes exist in the code but are deliberately switched off for this training version — that's intentional, not a missing feature.

## Mapping-first AOI selection

You'll see the scene footprint before you're asked for an AOI. The lat/lon bounding-box check is still what actually decides validity here — the map just helps you picture it.

In [ ]:
from pathlib import Path
import numpy as np

from nisar_utils.bootstrap import setup_workshop
WORKSHOP_ROOT = setup_workshop()

from nisar_utils.config import load_config
from nisar_utils.workflow import (
    build_profile,
    resolve_frequency,
    resolve_terms,
)
from nisar_utils.gcov import open_gcov, get_grid_coordinates
from nisar_utils.spatial import (
    coordinate_to_index,
    get_window_extent,
)

cfg = load_config()
NISAR_FILE = Path(cfg["nisar_file"])

if not NISAR_FILE.exists():
    raise FileNotFoundError(f"NISAR file not found: {NISAR_FILE}")

profile = build_profile(cfg)
freq = resolve_frequency(cfg, profile)
terms, diagonal_terms, off_diagonal_terms = resolve_terms(profile, freq)

grid = f"{profile.gcov_root}/grids/{freq}"

print("=" * 70)
print("NISAR GCOV SPATIAL SUBSETTING")
print("=" * 70)
print("File:", NISAR_FILE)
print("SAR family:", profile.sar_family)
print("Band:", profile.band)
print("Level:", profile.product_level)
print("Product:", profile.product_type)
print("Frequency:", freq)
print("Grid:", grid)
print("Product EPSG:", profile.epsg)


In [ ]:
with open_gcov(NISAR_FILE) as f:
    x, y = get_grid_coordinates(f, grid)

grid_xmin = float(np.nanmin(x))
grid_xmax = float(np.nanmax(x))
grid_ymin = float(np.nanmin(y))
grid_ymax = float(np.nanmax(y))

print("Actual NISAR GCOV grid coverage in native projected CRS:")
print(f"  X: {grid_xmin} to {grid_xmax}")
print(f"  Y: {grid_ymin} to {grid_ymax}")
print("Grid size:", len(y), "rows x", len(x), "columns")


In [ ]:
def validate_projected_aoi(
    xmin, xmax, ymin, ymax,
    gxmin, gxmax, gymin, gymax
):
    """Return status and overlap bounds.

    status:
      inside  = AOI fully inside product grid
      partial = AOI overlaps but extends outside product grid
      outside = AOI has no overlap
    """
    if xmin >= xmax or ymin >= ymax:
        raise ValueError(
            "Invalid AOI: minimum coordinate must be smaller than maximum."
        )

    overlap_x = min(xmax, gxmax) - max(xmin, gxmin)
    overlap_y = min(ymax, gymax) - max(ymin, gymin)

    if overlap_x <= 0 or overlap_y <= 0:
        return "outside", None

    overlap = (
        max(xmin, gxmin),
        min(xmax, gxmax),
        max(ymin, gymin),
        min(ymax, gymax),
    )

    partial = (
        xmin < gxmin or xmax > gxmax or
        ymin < gymin or ymax > gymax
    )

    return ("partial" if partial else "inside"), overlap


def show_aoi_status(status, requested, coverage):
    print("=" * 70)

    if status == "inside":
        print("AOI STATUS: VALID")
        print("=" * 70)
        print("The requested AOI lies completely within NISAR GCOV coverage.")

    elif status == "partial":
        print("AOI STATUS: PARTIALLY OUTSIDE NISAR COVERAGE")
        print("=" * 70)
        print("The requested AOI overlaps the product but extends outside")
        print("the actual NISAR GCOV grid.")

    else:
        print("AOI STATUS: OUT OF BOUNDS")
        print("=" * 70)
        print("The requested AOI has no overlap with the actual")
        print("NISAR GCOV grid coverage.")

    print("\nRequested product-CRS AOI:")
    print("  X:", requested[0], "to", requested[1])
    print("  Y:", requested[2], "to", requested[3])

    print("\nNISAR product coverage:")
    print("  X:", coverage[0], "to", coverage[1])
    print("  Y:", coverage[2], "to", coverage[3])


In [ ]:
# ================================================================
# ACTUAL NISAR GEOGRAPHIC EXTENT — WGS84 LAT/LON
# ================================================================
# Display the actual scene coverage before asking the user for AOI.
# The extent is derived from the selected GCOV grid and its actual EPSG.

if profile.epsg is None:
    raise ValueError(
        "Product EPSG could not be determined; cannot report "
        "the NISAR extent in latitude/longitude."
    )

try:
    from pyproj import Transformer
except ImportError as exc:
    raise ImportError(
        "Latitude/longitude extent reporting requires pyproj "
        "in the active environment."
    ) from exc

to_wgs84 = Transformer.from_crs(
    f"EPSG:{int(profile.epsg)}",
    "EPSG:4326",
    always_xy=True,
)

# Transform all four corners rather than transforming only min/max
# coordinates. This is safer for projected CRSs.
grid_corners_xy = [
    (grid_xmin, grid_ymin),
    (grid_xmin, grid_ymax),
    (grid_xmax, grid_ymin),
    (grid_xmax, grid_ymax),
]

grid_corners_ll = [
    to_wgs84.transform(px, py)
    for px, py in grid_corners_xy
]

scene_lon_min = min(p[0] for p in grid_corners_ll)
scene_lon_max = max(p[0] for p in grid_corners_ll)
scene_lat_min = min(p[1] for p in grid_corners_ll)
scene_lat_max = max(p[1] for p in grid_corners_ll)

print("=" * 70)
print("ACTUAL NISAR GCOV GEOGRAPHIC COVERAGE — WGS84")
print("=" * 70)
print("Frequency :", freq)
print("EPSG      :", profile.epsg)
print()
print(
    f"Longitude : {scene_lon_min:.6f}° to {scene_lon_max:.6f}°"
)
print(
    f"Latitude  : {scene_lat_min:.6f}° to {scene_lat_max:.6f}°"
)
print()
print("Use this geographic extent as a reference before")
print("entering the requested Latitude/Longitude AOI.")
print("=" * 70)


In [ ]:
from nisar_utils.mapping import plot_scene_overview,folium_scene_map
print("MAP CONTEXT — full NISAR scene")
plot_scene_overview(x,y,profile.epsg,title=f"NISAR {profile.sar_family} {freq} — AOI Reference")
scene_map=folium_scene_map(x,y,profile.epsg,title="NISAR Footprint — AOI Reference",add_draw=True)
scene_map


In [ ]:
# Optional interactive AOI drawing. After drawing, run this cell again to read the selected bounds.
from nisar_utils.mapping import interactive_aoi_map
try:
    aoi_map = interactive_aoi_map(x,y,profile.epsg)
    print("Draw a rectangle over the area you want to analyse.")
    print("The scientific Lat/Lon AOI entry below remains the authoritative validation step.")
except Exception as e:
    print("Interactive AOI map unavailable:",e)


In [ ]:
# ================================================================
# OPTION 1 — LATITUDE / LONGITUDE BOUNDING BOX (DEFAULT)
# ================================================================

lon_min = float(input("Longitude minimum (degrees): "))
lon_max = float(input("Longitude maximum (degrees): "))
lat_min = float(input("Latitude minimum (degrees): "))
lat_max = float(input("Latitude maximum (degrees): "))

if not (-180 <= lon_min <= 180 and -180 <= lon_max <= 180):
    raise ValueError("Longitude values must be within -180 to +180 degrees.")

if not (-90 <= lat_min <= 90 and -90 <= lat_max <= 90):
    raise ValueError("Latitude values must be within -90 to +90 degrees.")

if lon_min >= lon_max or lat_min >= lat_max:
    raise ValueError(
        "Invalid geographic bounding box: minimum must be smaller than maximum."
    )

if profile.epsg is None:
    raise ValueError(
        "Product EPSG could not be determined; Lat/Lon subsetting "
        "requires the actual product CRS."
    )

try:
    from pyproj import Transformer
except ImportError as exc:
    raise ImportError(
        "Latitude/longitude subsetting requires pyproj "
        "in the active environment."
    ) from exc

transformer = Transformer.from_crs(
    "EPSG:4326",
    f"EPSG:{int(profile.epsg)}",
    always_xy=True,
)

corners = [
    transformer.transform(lon_min, lat_min),
    transformer.transform(lon_min, lat_max),
    transformer.transform(lon_max, lat_min),
    transformer.transform(lon_max, lat_max),
]

req_xmin = min(p[0] for p in corners)
req_xmax = max(p[0] for p in corners)
req_ymin = min(p[1] for p in corners)
req_ymax = max(p[1] for p in corners)

requested_aoi = (req_xmin, req_xmax, req_ymin, req_ymax)
coverage = (grid_xmin, grid_xmax, grid_ymin, grid_ymax)

print("\nRequested AOI after transformation to product CRS:")
print("  X:", req_xmin, "to", req_xmax)
print("  Y:", req_ymin, "to", req_ymax)

status, overlap = validate_projected_aoi(
    *requested_aoi,
    *coverage,
)

show_aoi_status(status, requested_aoi, coverage)

if status == "outside":
    raise ValueError(
        "AOI is completely outside NISAR GCOV coverage. "
        "Processing stopped before subset extraction."
    )

if status == "partial":
    print("\nOverlapping portion:")
    print("  X:", overlap[0], "to", overlap[1])
    print("  Y:", overlap[2], "to", overlap[3])

    answer = input(
        "\nContinue with the overlapping portion only? [y/N]: "
    ).strip().lower()

    if answer not in {"y", "yes"}:
        raise ValueError(
            "Processing cancelled because the requested AOI "
            "extends outside NISAR GCOV coverage."
        )

    req_xmin, req_xmax, req_ymin, req_ymax = overlap
    print("Continuing with the overlapping portion.")
else:
    print("\nAOI accepted without clipping.")


In [ ]:
# Convert the validated projected AOI to row/column indices.
# This uses the existing v7.5 coordinate_to_index() function.

r0, r1, c0, c1 = coordinate_to_index(
    x, y,
    req_xmin, req_xmax,
    req_ymin, req_ymax
)

# Persist the validated AOI and the exact native NISAR subset indices.
# This is downstream hand-off information only; the existing AOI validation
# and subsetting logic above remains unchanged.
from nisar_utils.config import save_session, load_config

# Module 02 is authoritative for frequency selection. Preserve the selected
# frequency while adding Module 06's validated AOI/subset information.
# load_config() is intentionally used here so existing session state is
# carried forward instead of being replaced by an AOI-only dictionary.
try:
    existing_cfg = load_config(require_file=False)
except TypeError:
    existing_cfg = load_config()
    
requested_aoi_wgs84 = {
    "xmin": float(lon_min),
    "xmax": float(lon_max),
    "ymin": float(lat_min),
    "ymax": float(lat_max),
}
session_update = {
    "nisar_file": str(NISAR_FILE.resolve()),
    "aoi": requested_aoi_wgs84,
    "default_aoi": requested_aoi_wgs84,
    "validated_projected_aoi": {
        "xmin": float(req_xmin),
        "xmax": float(req_xmax),
        "ymin": float(req_ymin),
        "ymax": float(req_ymax),
    },
    "spatial_subset": {
        "r0": int(r0),
        "r1": int(r1),
        "c0": int(c0),
        "c1": int(c1),
    },
    "spatial_subset_mode": "latlon",
}

# Explicitly carry forward Module 02's selected frequency if present.
if existing_cfg.get("default_frequency") is not None:
    session_update["default_frequency"] = existing_cfg["default_frequency"]

save_session(session_update)

print("AOI/session state saved: PASS")
print("NISAR file preserved: PASS")
if session_update.get("default_frequency") is not None:
    print(
        "Selected frequency preserved: PASS ->",
        session_update["default_frequency"]
    )
else:
    print(
        "Selected frequency preserved: NOT PRESENT "
        "(run Module 02 first when multiple frequencies exist)"
    )

print("\nFinal Lat/Lon subset:")
print("  Row indices:", r0, r1)
print("  Column indices:", c0, c1)
print("  Size:", r1-r0, "rows x", c1-c0, "columns")
print(
    "  Native projected extent:",
    get_window_extent(x, y, r0, r1, c0, c1)
)


### Training note

Worth repeating: lat/lon bounding box is the only subsetting mode live in this version. If you're used to pixel/row-column or projected X/Y from other SAR tools, you won't find them here — on purpose.

In [ ]:
print("=" * 70)
print("MODULE 06 — SPATIAL SUBSETTING VALIDATION")
print("=" * 70)
print("Active/default option : Latitude / Longitude bounding box")
print("Pixel/row-column      : DISABLED")
print("Projected X/Y         : DISABLED")
print("Actual WGS84 extent   : Printed before AOI input")
print("Out-of-bounds check   : Enabled")
print("Partial-overlap flag  : Enabled")
print("Validation precedes   : subset/window extraction")
print("MODULE 06 STATUS      : PASS")


### Handing the AOI downstream

Once validation succeeds, both the lat/lon AOI and the resulting pixel indices (`r0`, `r1`, `c0`, `c1`) are saved to the session. Every module from here on reads those indices back rather than asking you to redefine the AOI.

## Final AOI map
One last plot, showing the validated AOI next to the full scene footprint side by side.

In [ ]:
from nisar_utils.mapping import folium_scene_map
final_map=folium_scene_map(x,y,profile.epsg,title="Validated NISAR AOI",aoi={"lon_min":lon_min,"lon_max":lon_max,"lat_min":lat_min,"lat_max":lat_max})
final_map
